# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# **Ranking (with a scoring layer underneath).**

The lane's actual output is a queue — "review this page before that one" — not a single yes/no bucket. Ranking is the right framing because what matters isn't just "is this page at risk" (that would be classification) but how urgently, relative to every other page competing for the same reviewer's limited hours.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_task_type = "ranking (via a continuous risk/opportunity score)"
print(lane_task_type)

ranking (via a continuous risk/opportunity score)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

What I'd predict: whether a page's clicks are declining — using trend_direction `(or trend_pct < 0)`, which compares` clicks_last_30d` against `clicks_prev_30d.`
Where the label comes from: a defined rule, not an observed real-world outcome. Nobody in this data actually refreshed a page and we watched what happened next — we only see two 30-day windows next to each other. So "declining" here means "clicks went down between two windows I can see," not "this page needs a refresh and refreshing will fix it."

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("target: trend_direction == 'down'")
print("source: rule-based comparison of two existing windows, not a true outcome label")

target: trend_direction == 'down'
source: rule-based comparison of two existing windows, not a true outcome label


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@20 (or Precision@50, matching the size of a review queue a reviewer could realistically get through in a week). Plain meaning: of the top 20 pages my model flags, how many are actually declining pages worth a look? "Good" means beating the simple rule-based baseline on this same number — e.g., if the baseline's Precision@20 is 0.60, the model needs to clear that, not just look busy. I picked this over overall accuracy because accuracy on 30,000 mostly-fine pages is easy to fake by predicting "stable" for everything — precision on the top of the queue is the number that actually matters to the person using it.

***success metric as future work***

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
success_metric = "Precision@20"
good_means = "exceeds the rule-based baseline's Precision@20 on the same holdout window"
print(success_metric, "->", good_means)

Precision@20 -> exceeds the rule-based baseline's Precision@20 on the same holdout window


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("One row =", df[["content_id", "client_id"]].drop_duplicates().shape[0], "unique pages")
df.head(3)

Shape: (30000, 44)
One row = 30000 unique pages


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag if declining AND page-one" works for the obvious cases, but breaks down because the combination of signals matters differently depending on the page. A small drop in clicks on a low-search_volume, high-competition page might be normal noise; the same drop on a high-search_volume page in a different content_type might be an early warning. A rule has to pick one threshold for everyone; a model can weigh avg_position, ctr, word_count, content_age_days, and competition_level together, and let the weighting shift depending on how they interact — something an if-statement can't do without turning into hundreds of nested conditions that still wouldn't generalize to a new client's pages.

**One-line summary: a rule needs a human to spell out every combination in advance; a model learns which combinations matter by looking at the data.**

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.